# 黑马头条项目｜第13篇：项目实战——文章评论、自媒体评论管理与图文统计

> 学习目标：依据需求和接口完成评论/回复闭环，理解 MongoDB 文档与点赞关系、自媒体作者的管理权限、评论开关与级联清理，以及按作者和时间范围查询图文统计。

课程目录名为 `day13-项目实战[2天]`，实战说明要求分组、两天完成三个模块。本篇覆盖这个目录的全部三个模块；资料没有明确按第13/14天拆分模块，因此下面的两天开发顺序属于学习建议，不是另造一份课程日程。

本篇已完整阅读网页预览中的《实战说明.md》、三个模块的需求说明和接口文档，以及公共资料中的 `leadnews-comment.sql` MongoDB 脚本。共整理15个接口：APP评论/回复6个、自媒体评论管理7个、图文统计2个。未读取参考工程、视频、图片资产和 `wemedia-web-v2.zip` 内部源码；独立 `.conf` 点击后未显示正文，Nginx配置来自已读实战说明。原图未嵌入。

“课程约定”描述原资料，“工程补充/设计示意”用于解释正确性与联调边界，不代表参考代码已经实现。代码均为不可直接照搬的学习片段；未导入数据库、启动服务、调用业务接口或执行删除操作。

资料：[黑马头条项目百度网盘分享](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 三个模块与业务闭环

| 模块 | 课程要求 | 核心结果 |
|---|---|---|
| APP文章评论 | 发表评论、查询、点赞/取消；回复的保存、查询、点赞/取消 | 当前用户操作后，计数与回显一致 |
| 自媒体评论管理 | 已发布文章列表、评论开关、查看评论和回复、作者回复/点赞、删除评论/回复 | 作者仅管理自己文章下的评论 |
| 自媒体图文统计 | 指定时间查询汇总、分页查询文章指标 | 汇总和列表使用相同作者与时间条件 |

~~~text
APP登录 → 文章详情 → 发表评论 → MongoDB评论
                          ↓              ↓
                     文本垃圾检测     回复 / 点赞记录
                                         ↓
自媒体登录 → 自媒体微服务 → 校验文章归属 → 管理评论
                    ↓
              图文汇总 + 文章指标分页
~~~

课程规定新建评论模块 `heima-leadnews-comment`，评论相关数据都保存到 MongoDB；自媒体管理功能必须先经过自媒体微服务，统计接口也定义在自媒体微服务。不要让前端直接连接数据库。

**工程补充：** 评论领域可以由评论服务提供内部能力，自媒体服务负责作者身份、文章归属校验和调用编排。具体 Feign 路径、服务端口与注册名应对照工程配置，资料未给出时不自行当作课程事实。


## 2. 实战资料与联调准备

| 已查看的公共资料 | 实际内容或读取情况 |
|---|---|
| `评论的表-导入mongodb中/leadnews-comment.sql` | 已读完整预览；内容是 MongoDB 集合脚本，含 `drop/createCollection/insert` |
| `nginx配置/heima-leadnews-wemedia.conf` | 已看到文件；独立预览没有正文，使用实战说明中的配置 |
| `自媒体前端项目/wemedia-web-v2.zip` | 已看到文件名，未解压/读取 |
| 两个自媒体模块各自的“资料”目录 | 网页显示“没有数据”，不能据此补造表结构 |

开始实现前，核对 APP网关、自媒体网关、自媒体服务、文章服务、评论服务、MongoDB和已有用户认证链路。自媒体前端要使用课程提供的新版本，并将Nginx静态根目录指向实际解压位置。

**联调准备建议：** 准备文章作者A、作者B、普通APP用户U，以及A/B各自发布的文章。建立跨服务文章ID和作者ID对应关系；使用合成评论文本，避免拿真实数据测试清理操作。所有数据库导入都应先检查目标数据库和脚本内容。


## 3. MongoDB数据模型：四个集合

脚本头部标明数据库 `leadnews-comment`，示例MongoDB版本为4.2.1。虽然文件扩展名是 `.sql`，正文调用的是 `db.getCollection(...)`、`ObjectId`、`NumberLong`、`ISODate`；它不是MySQL建表脚本。

| 集合 | 已读脚本中的主要字段 | 关系 |
|---|---|---|
| `ap_comment` | `_id`、`authorId`、`authorName`、`entryId`、`type`、`content`、`likes`、`reply`、`flag`、`createdTime`、`_class` | `entryId`关联文章 |
| `ap_comment_like` | `_id`、`authorId`、`commentId`、`_class` | 用户对评论的点赞关系 |
| `ap_comment_repay` | `_id`、`authorId`、`authorName`、`commentId`、`content`、`likes`、`createdTime`、`updatedTime`、`_class` | `commentId`关联父评论 |
| `ap_comment_repay_like` | `_id`、`authorId`、`commentRepayId`、`_class` | 用户对回复的点赞关系 |

导入样例中 `_id` 为 ObjectId，文章 `entryId` 为 NumberLong，作者ID为 NumberInt，关联评论/回复ID保存为字符串。API响应将文档主键展示为 `id`。应用查询必须与实际 BSON 类型一致；字符串值相同不意味着 String 和 ObjectId/Long 能直接匹配。

脚本的点赞样例没有 `operation` 字段：可以用“存在记录=已点赞，删除记录=取消”实现活动关系；也可以设计状态记录，但不能同时混用两种语义。APP接口仍用 `operation` 表示动作和回显。

**工程补充：** 脚本先删除再建立四个集合，会清空已有数据；此处只阅读，没有执行。脚本里 `_class` 存在不同包名，映射实体时需核对Spring Data类型信息与包迁移，不能把样例包名视为必须新建的Java包。


## 4. Spring Data MongoDB映射与类型检查

以下为**设计示意**，展示集合名与主键映射，未读取完整参考实体：

~~~java
@Document(collection = "ap_comment")
class ApComment {
    @Id
    private String id;       // 应核对当前版本是否将合法字符串映射为ObjectId
    private Long entryId;    // 对应NumberLong文章ID
    private Integer authorId;
    private String authorName;
    private String content;
    private Integer likes;
    private Integer reply;
    private Date createdTime;
}
~~~

Spring Data的字符串 `@Id` 在可转换时可能按ObjectId处理；手写Mongo查询则需要显式使用正确类型。不要仅看到API的 `id` 是字符串，就假设库中 `_id` 也是字符串。关联字段 `commentId/commentRepayId` 在脚本中是String，要分别处理。

配置数据库名与实际导入目标保持一致；MongoDB URI中的账号和密码使用外部配置。该模块需要Mongo依赖、Nacos配置、服务发现和网关路由，但具体版本应沿用项目依赖管理。

**工程补充：** 超出JavaScript安全整数范围的文章Long ID对前端输出为字符串；Mongo内部仍存Long，服务端严格解析输入。ObjectId和Long文章ID是两个独立概念，不可互相替换。


## 5. APP评论和回复：六个接口总表

全部使用 `POST`、`application/json` 请求；统一响应包含 `host/code/errorMessage/data`。

| 功能 | 原接口路径 | 请求字段/DTO |
|---|---|---|
| 保存评论 | `/api/v1/comment/save` | `articleId` Long、`content` String；`CommentSaveDto` |
| 查询评论 | `/api/v1/comment/load` | `articleId`、`minDate` Date；`CommentDto` |
| 评论点赞/取消 | `/api/v1/comment/like` | `commentId` String、`operation` Short；`CommentLikeDto` |
| 保存回复 | `/api/v1/comment_repay/save` | `commentId`、`content`；`CommentRepaySaveDto` |
| 查询回复 | `/api/v1/comment_repay/load` | `commentId`、`minDate`、`size` int；`CommentRepayDto` |
| 回复点赞/取消 | `/api/v1/comment_repay/like` | `commentRepayId`、`operation`；`CommentRepayLikeDto` |

课程路径使用 `repay`，不要擅自改成 `reply`，否则现有前端请求会404。接口文档将不少字段标记为“非必须”，但业务仍必须验证目标ID、内容和操作值；自动生成文档的布尔标记不能代替需求校验。

评论列表请求示例还有 `index`，参数表没有解释其含义；不能据此断言它就是页码。应对照前端请求及DTO确认。回复接口明确包含 `size`，默认条数可遵循需求中的10条，并限制最大值。


## 6. 发表评论：登录、140字、内容检测

**课程约定：** 在文章详情输入评论；必须登录；内容不超过140字；进行文本垃圾检测。所有评论数据存MongoDB。

~~~json
{
  "articleId": "1428570910251683841",
  "content": "这篇文章帮助我理解了微服务之间的调用。"
}
~~~

上面的字符串文章ID是**前端精度保护示意**，原接口schema为Long，服务端是否接收数字字符串要按实际JSON转换配置联调。

**建议实现顺序：** 获取可信登录身份 → 校验目标文章及评论状态 → 验证内容 → 调用文本检测 → 构造评论 → 保存 → 更新相关计数 → 返回结果。作者ID、昵称、创建时间由服务端决定；初始 `likes=0`、`reply=0`。

“打开评论允许任何用户评论”仍需同时遵守“发表评论必须登录”，不能解释成游客可以写入。前端限制长度用于交互提示，后端必须重复校验。

**工程补充：** 明确140字的计数规则，Java `String.length()`计的是UTF-16代码单元，emoji可能占两个；如按Unicode码点，可用 `codePointCount`。空白内容、目标不存在、已关闭评论、检测拒绝/超时应有明确响应；超时策略不能默认等同检测通过。


## 7. 查询评论：时间倒序与点赞回显

**课程约定：** 按当前文章查询，创建时间倒序，分页默认10条；展示内容、作者、点赞数、回复数、时间。当前登录用户点赞的评论要高亮。

原请求示例为 `articleId/index/minDate`，明确的分页条件是 `minDate`（上一批数据中的最小时间）。返回列表的 `operation=0` 表示当前用户已赞，`operation=null` 表示未赞。

~~~text
第一页：entryId=文章ID，按createdTime降序，limit 10
下一页：entryId=文章ID 且 createdTime < minDate，继续降序
回显：批量查“当前用户 + 本页commentId集合”的点赞记录
      命中 → operation=0；未命中 → operation=null
~~~

接口示例还包含 `id/authorId/authorName/entryId/channelId/type/content/image/likes/reply/flag/longitude/latitude/address/ord/createdTime/updatedTime`，多数可为空；这些是响应示例字段，不应据此强制采集位置。优先满足页面实际展示字段。

**工程补充：** 仅用时间 `< minDate` 时，同毫秒的评论可能漏读；推荐稳定按 `(createdTime, _id)` 降序，用双字段游标。在原DTO未扩展前，记录这一限制并验证边界。每条评论单独查点赞会造成N+1查询，应按本页ID批量查。

游客能否读列表要按实际网关规则确定；若允许，游客回显统一未赞，不能使用其他用户的状态。


## 8. 评论点赞：关系记录与计数共同变化

请求 `commentId/operation`，课程定义 `0=点赞，1=取消点赞`；响应 `data.likes` 是最新点赞数。一个总数字无法判断某个用户是否已经点赞，因此必须保存用户和评论的关系。

| 原状态 | 请求动作 | 应产生的变化（工程补充） |
|---|---|---|
| 未赞 | 点赞0 | 建立关系，likes加1 |
| 已赞 | 点赞0 | 幂等返回，不重复加1 |
| 已赞 | 取消1 | 删除/更新关系，likes减1 |
| 未赞 | 取消1 | 幂等返回，不产生负数 |

~~~text
点赞：只在“关系从不存在变为存在”时增加计数
取消：只在“关系从存在变为不存在”时减少计数
~~~

**工程补充：** 建立用户+目标唯一索引，阻止并发产生重复关系；计数更新用MongoDB `$inc`，不要“读取旧值再保存新值”。但关系记录和评论文档是两个文档，单条 `$inc` 并不能保证整个点赞事务原子。

可在支持事务的Mongo部署中将关系和计数放入同一事务；或用唯一事件、重试和定期对账恢复。重复键异常、关系删除数量为0时，不应继续更改计数。`likes>=0`只是防护，不替代关系一致性。

评论点赞与第09篇文章点赞是不同业务目标；不得向“文章获赞数”直接累加一条评论的点赞。


## 9. 回复：父评论关系、回复数和独立点赞

**课程约定：** 发表回复与评论要求一致，保存成功后更新父评论回复数；回复点赞与评论点赞一致；回复列表与评论列表查询要求一致。

| 接口 | 关键校验与行为 |
|---|---|
| `comment_repay/save` | 登录；父评论存在；所属文章开放评论；内容≤140字并检测；保存回复后父评论 `reply+1` |
| `comment_repay/load` | 按 `commentId` 过滤，使用 `minDate/size` 查询；回显当前用户点赞状态 |
| `comment_repay/like` | 目标是 `commentRepayId`，更新回复的likes和 `ap_comment_repay_like` |

回复记录的 `commentId` 是Mongo父评论ID，不是文章ID。回复ID也不能传到普通评论点赞接口。

回复列表的 `operation=0` 已赞、`null` 未赞，保持APP约定。回复文档展示作者、内容、点赞、创建时间；排序和游标边界按评论同样处理。

**工程补充：** 回复保存与父评论 `reply` 计数涉及多文档；失败重试需要事务或幂等请求标识，不能因为计数失败就重复插入回复。父评论可能与作者删除操作并发，要防止“评论已删但回复刚写入”的孤儿记录。

接口只给出一层 `commentId` 关系，未定义无限级楼中楼，不要凭空扩成任意层级回复树。


## 10. 查询索引与原子更新示意

以下为**工程补充的索引建议**，并非已读脚本中声明的索引；应先检查历史数据、实际查询和执行计划再创建。

| 集合 | 建议索引 | 目的 |
|---|---|---|
| `ap_comment` | `{entryId:1, createdTime:-1, _id:-1}` | 文章评论倒序和游标 |
| `ap_comment_repay` | `{commentId:1, createdTime:-1, _id:-1}` | 回复列表和按父评论清理 |
| `ap_comment_like` | 用户身份 + `commentId` 唯一联合索引 | 幂等点赞、批量回显 |
| `ap_comment_repay_like` | 用户身份 + `commentRepayId` 唯一联合索引 | 幂等回复点赞 |

~~~java
// 示意：单个文档原子加计数；不包含关系写入事务。
Query query = Query.query(Criteria.where("_id").is(commentObjectId));
Update update = new Update().inc("likes", 1);
mongoTemplate.updateFirst(query, update, "ap_comment");
~~~

`commentObjectId`应由已验证的合法ID转换，错误ID不能抛原始数据库异常给前端。查询结果为0条需区别目标不存在、类型不匹配和并发删除。

**身份补充：** APP用户ID与自媒体用户ID属于不同域，数字相同不一定是同一人。如果共同写点赞集合，需统一映射身份或增加 `subjectType/subjectId` 等域信息，再基于完整身份建唯一索引。该扩展会改变课程样例模型，必须同时迁移查询和历史数据。


## 11. 自媒体评论管理：七个接口总表

课程要求全部功能先经过自媒体微服务。下表保留原文路径，不默默修正前缀。

| 功能 | Method | 原文Path | 入参 |
|---|---|---|---|
| 删除评论 | DELETE | `/wemedia/api/v1/comment/manage/del_comment/{commentId}` | 路径commentId |
| 删除回复 | DELETE | `/api/v1/comment/manage/del_comment_repay/{commentRepayId}` | 路径commentRepayId |
| 作者回复 | POST | `/api/v1/comment/manage/comment_repay` | JSON：commentId、content |
| 评论开关 | POST | `/api/v1/comment/manage/update_comment_status` | JSON：articleId、operation |
| 已发布文章及评论概览 | POST | `/api/v1/comment/manage/find_news_comments` | JSON：beginDate、endDate、page、size |
| 某篇文章的评论和回复 | POST | `/api/v1/comment/manage/list` | JSON：articleId、page、size |
| 作者点赞/取消 | POST | `/api/v1/comment/manage/like` | JSON：commentId、operation |

点赞动作仍是 `0=点赞、1=取消`；开关动作则是 `0=关闭、1=开启`。两者字段同名而含义不同，要定义不同枚举。

管理接口的 `articleId` 在文档里以integer/number出现，但实际文章ID可能为Long，不能按Java int接收。评论/回复主键为String。日期和分页字段标为非必须，默认值、格式、范围要在项目中统一。


## 12. 管理文章列表：作者范围、时间范围、分页

需求要求按照时间倒序查询已发布文章，展示标题、评论状态、评论总量，支持分页和时间范围筛选。对应 `find_news_comments`，不是直接拉取评论集合。

| 响应字段 | 课程文档含义 |
|---|---|
| `data[].id` | 文章ID，示例为字符串 |
| `data[].title` | 文章标题 |
| `data[].comments` | 评论总量 |
| `data[].isComment` | boolean评论状态 |
| `data[].createdTime` | 创建时间 |
| `currentPage/size/total` | 分页信息，与统一响应字段同层 |

**设计示意：** 获取当前Wm用户 → 找到其已发布文章 → 应用日期范围 → 按时间稳定排序并分页 → 补齐APP文章ID、评论开关、评论数量 → 组装VO。

自媒体稿件ID与发布后的APP文章ID可能不同，必须按真实关联字段建立映射。评论集合 `entryId` 对应哪个ID需要查工程发布链路，不能用“数字都叫articleId”代替关联设计。

**工程补充：** `total`应为该作者、相同过滤条件下的文章总数。文档出现“总数100、每页10、文章评论数1000～9999”等示例约束，不能当业务固定值。确认需求所谓时间采用发布时间还是创建时间，保持列表和筛选一致。


## 13. 管理详情：评论与回复的嵌套返回

管理 `list` 请求 `articleId/page/size`；文档返回的 `data` 是数组，每项包含 `apComments` 评论对象和 `apCommentRepays` 回复数组。

~~~json
{
  "data": [
    {
      "apComments": {
        "id": "60c06eda4e71825b1b14f0f7",
        "content": "示例评论",
        "likes": 1,
        "reply": 1,
        "operation": 0
      },
      "apCommentRepays": [
        {
          "id": "60c2052bddf781003cc72e67",
          "commentId": "60c06eda4e71825b1b14f0f7",
          "content": "示例作者回复",
          "likes": 0
        }
      ]
    }
  ]
}
~~~

这是**结构示意**，省略身份、时间和外层响应字段。原表评论还包括 `authorId/authorName/entryId/channelId/type/image/flag/ord/createdTime/updatedTime/status`，回复包括作者和时间字段。

查询前先确认文章归当前作者所有。用本页评论ID批量查回复和当前作者点赞状态，再分组组装；避免每条评论单独执行多次查询。文档没有明确定义该接口的所有分页元信息，不应假定与 `find_news_comments` 完全相同。

**工程补充：** 一条评论可能有很多回复，直接嵌入全部回复会放大响应体。限制预览数量并另设加载能力属于接口扩展，需要前后端共同修改；不能说课程已规定这种设计。


## 14. 作者回复和点赞：不能信任客户端作者信息

需求允许作者对评论回复、点赞。分别对应 `comment_repay` 与管理 `like`；点赞返回 `data.likes`，操作0点赞、1取消。

~~~text
Wm登录身份 → commentId查父评论 → entryId查文章
           → 映射文章作者 → 验证当前Wm用户
           → 执行回复/点赞 → 返回最新状态
~~~

**工程补充：** 授权应在修改前完成，且覆盖所有管理入口。仅校验“已登录”不足以防止作者B删除、回复或关闭作者A文章下的评论。回复删除更需通过“回复 → 父评论 → 文章 → 作者”追溯归属。

作者回复身份从Wm认证上下文生成，不接受客户端提交 `authorId/authorName` 决定作者。APP端昵称与自媒体作者昵称之间如何映射，依项目账户关系处理。二者恰巧使用相同整数ID不能说明它们天然共享身份。

课程对APP回复明确140字和检测，自媒体需求只写“作者可以回复”；建议复用同样的内容验证和检测策略，并作为工程补充。管理列表的 `operation` 表注释是动作0/1，示例偏向1，没有像APP文档一样明确null回显；联调时确认未赞状态，不能静默替换原协议。


## 15. 删除评论与回复：关系清理和计数维护

需求允许作者删除评论和回复。原接口分别是 `del_comment/{commentId}` 和 `del_comment_repay/{commentRepayId}`。以下级联步骤为**工程补充**，用于避免孤儿数据，并非需求文档逐项给出的算法。

| 操作 | 建议同步处理 |
|---|---|
| 删除评论 | 评论本身、其所有回复、评论点赞关系、回复点赞关系；文章评论数按口径修正 |
| 删除一条回复 | 回复本身及其点赞关系；父评论reply按实际删除数量减1 |
| 重复删除 | 幂等，不重复减计数，不返回底层异常 |

~~~text
删父评论：先识别子回复ID → 清子回复点赞 → 清评论点赞
          → 清回复 → 清父评论 → 修正文章指标
删回复：定位父评论 → 校验归属 → 清点赞关系 → 删回复
        → 只有首次真实删除才减父评论reply
~~~

实际顺序与事务策略要结合并发写入设计：仅按上述顺序逐条执行，没有事务、锁定/删除标记或补偿机制，仍可能出现中途失败和并发插入。支持事务时可保护Mongo内部操作；跨Mongo/MySQL的文章计数需要另外设计可靠事件或对账。

“评论总量”是否包含回复未被需求明确说明。一般可区分主评论数与回复数，但须让详情、概览、文章指标、热点计分使用同一约定，不能默认减去所有回复后再额外减父评论。


## 16. 评论开关：关闭会清除既有评论

**课程约定：** 打开评论允许用户评论；关闭评论禁止评论，并清除该文章已有的全部评论。开关接口请求 `articleId/operation`，`0=关闭、1=开启`。文章概览返回boolean `isComment`。

状态字段存在哪张表/哪个集合、如何映射到APP文章配置，原文没有完整实现，应对照工程确认，不能直接臆造数据库列。

**设计示意：** 校验当前作者与文章归属 → 持久化关闭状态 → 阻止新的评论和回复写入 → 按文章清理评论、回复及两类点赞关系 → 校准评论数量 → 使各端读状态一致。

| 边界 | 应验证 |
|---|---|
| 清理期间刚好有人发表评论 | 不能出现已关闭后仍新建可见评论 |
| 关闭请求重复到达 | 清理/计数处理幂等 |
| 清理到一半失败 | 可恢复重试，不能只更新前端开关 |
| 再次开启 | 允许新评论；不会自动恢复已清除的旧评论 |
| 关闭后有旧详情页 | 后端拒绝写入，不能只靠按钮禁用 |

先检查状态、后插入评论仍存在检查与写入之间的竞争窗口。**工程补充：** 可用按文章协调写入、关闭版本、删除标记及可靠清理任务等方案；Mongo事务本身不能自动解决“状态在MySQL、评论在Mongo”的跨存储竞争。

这是资料中的业务要求，本篇没有对实际数据库执行关闭或删除。


## 17. 原资料中的接口差异与确认点

| 实际观察到的差异 | 联调处理 |
|---|---|
| 删除评论Path带 `/wemedia`，其他管理接口未带 | 区分前端入口、Nginx重写、网关前缀和Controller映射；检查是否剥离前缀 |
| APP示例body成功 `code=0`，管理/统计示例多为 `code=200` | HTTP状态200与JSON业务code分开；核对各模块实际ResponseResult和前端判断 |
| 评论load示例有index，参数表只有articleId/minDate | 检查实际DTO和前端请求；index含义未给出 |
| APP `operation=0/null` 是明确点赞回显；管理表给0/1动作注释 | 确认管理列表未赞/取消的显示协议 |
| 图文需求文字重复写“点赞量”，且“图片发布量”用词不一致 | 汇总字段以实际接口 `likesNum/collectNum/publishNum` 为依据 |
| 接口schema使用number/integer的文章ID | 以实际Long和前端精度为准，不能缩成int |

**排查路径示意：** 浏览器请求URL → Nginx重写后的URL → 网关路由/StripPrefix → 微服务Controller。记录真实请求与响应后再决定修正位置；不要为了匹配一个接口而给所有请求重复追加 `/wemedia`。

原文成功响应有时 `errorMessage=null`，有时为“操作成功”，`host`也可为空；不能把文案或host为空当作失败判断。不要将自动导出样例的最大值/最小值/枚举当成正式业务限制。


## 18. 图文统计：两个GET接口与字段映射

需求要求当前自媒体用户按时间范围查看统计和文章指标列表。两个接口均定义在自媒体微服务，用query传参。

| 接口 | 请求参数 | 返回 |
|---|---|---|
| `GET /api/v1/statistics/newsDimension` | beginDate、endDate，String，可选 | data.likesNum、collectNum、publishNum |
| `GET /api/v1/statistics/newsPage` | beginDate、endDate、page、size，可选 | data文章数组；currentPage、size、total |

| 返回字段 | 含义与注意 |
|---|---|
| `likesNum` | 文章点赞量汇总 |
| `collectNum` | 文章收藏量汇总 |
| `publishNum` | 发布量；依据需求按文章发布数量理解，具体状态集合仍需确认 |
| `data[].id/title` | 单篇文章ID（示例String）/标题 |
| `data[].likes/collection/comment/views` | 文章点赞/收藏/评论/阅读数量 |

汇总接口没有返回 `viewsNum/commentNum`，也没有每日曲线数组，不应凭空添加为原接口内容。分页字段 `comment` 是单数；管理概览为 `comments`；Mongo评论里的 `reply` 是父评论回复数，三者要分别映射。

~~~json
{
  "code": 200,
  "errorMessage": "操作成功",
  "host": null,
  "data": {"likesNum": 104, "collectNum": 814, "publishNum": 13}
}
~~~

以上数值来自原接口示例，仅用于说明结构，不是用户项目的真实统计数据。


## 19. 统计查询：作者隔离、文章映射、汇总与分页

**设计示意：** 从可信身份得到当前Wm用户 → 筛出其符合时间条件的已发布稿件 → 映射到APP文章 → 批量获取文章计数 → 汇总或分页返回。

汇总要针对全部符合条件的文章；不能只把当前页的10篇文章相加。分页的 `total`也要使用同样的作者、日期、发布状态过滤。空结果汇总返回0，分页返回空数组和total=0。

| 查询要点 | 原因 |
|---|---|
| 当前作者由服务端确定 | 防止靠传作者ID查看他人统计 |
| 显式关联Wm稿件与APP文章 | 两端主键和作者身份可能不同 |
| 批量读取文章计数 | 避免逐篇Feign请求或逐行数据库查询 |
| 空值转换为0，聚合使用Long | 历史字段可空，累计数量可超过int |
| 稳定排序和有界分页 | 排序同值时避免跳页；size不无限放大 |

概念性聚合可理解为：`publishNum=count(符合条件的发布记录)`、`likesNum=sum(其文章点赞数)`、`collectNum=sum(其文章收藏数)`。这是一种**补充实现口径**，原资料没有给出SQL、Mapper或服务调用代码，不代表已经验证的课程算法。

跨服务数据不能直接假装成单库JOIN。可选择内部批量API、受控的统计读模型或异步同步，具体方案需结合项目架构。重复关联记录会放大统计，必须明确一稿与APP文章的关联基数。


## 20. 日期筛选与统计口径：累计数不等于期间新增数

原接口只给 `beginDate/endDate` 字符串，没有说明格式、时区、筛选的是发布时间还是行为发生时间，也没有提供按天的行为明细。需求文字“时间范围内数量”因此需要进一步落实口径。

| 口径 | 数据要求 | 示例 |
|---|---|---|
| 期间发布的文章，截至当前累计的点赞/收藏 | 文章发布时间 + 当前计数 | 昨天发的文章今天新增10赞，仍会计入昨天文章集合的累计量 |
| 期间发生的点赞/收藏行为 | 有行为时间的事件/日聚合 | 昨天文章今天10赞计入今天新增量 |
| 截至某日的历史累计快照 | 历史快照或可重放事件 | 查看上月末不能只读今天的当前值 |

只读取 `ap_article` 等当前累计计数不能还原某段时间内新增了多少点赞，也不能自动得到历史快照。将计数表筛选文章创建时间后求和属于第一种口径，应在实现和页面文案中明确。

**工程补充：** 若输入按日期筛选，例如2026-10-01至2026-10-07，可转成业务时区的 `[10月1日00:00, 10月8日00:00)`，避免用23:59:59遗漏毫秒数据。校验起止顺序、非法格式、仅给一端和范围上限，日期转换规则在前后端统一。

若要求行为期间报表，需要额外维护有时间维度的事件/聚合表、去重和延迟处理规则，属于扩展设计，不能宣称课程已经给出该表。


## 21. 新版自媒体前端与Nginx的GET参数转发

实战说明要求替换自媒体前端，公共资料提供 `wemedia-web-v2.zip`。新版有GET多参数请求，需要保留query。已读说明中的核心配置如下（本地静态目录改成示意路径）：

~~~nginx
upstream heima-wemedia-gateway {
    server localhost:51602;
}
server {
    listen 8802;
    location / {
        root D:/workspace/wemedia-web-v2/;
        index index.html;
    }
    location ~ /wemedia/MEDIA/(.*) {
        proxy_pass http://heima-wemedia-gateway/$1?$args;
        proxy_set_header HOST $host;
        proxy_pass_request_body on;
        proxy_pass_request_headers on;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    }
}
~~~

`$1`是正则捕获的后半段路径，`$args`是原请求查询参数字符串，不是POST请求体。在这个使用变量重新构造URI的配置中显式拼上query，才能保留 `beginDate/endDate/page/size`。

~~~text
前端示意：/wemedia/MEDIA/api/v1/statistics/newsPage?beginDate=...&page=1&size=10
代理示意：/api/v1/statistics/newsPage?beginDate=...&page=1&size=10
~~~

再由网关按实际路由转到自媒体服务。不能因此断言所有 `proxy_pass` 形式都会默认丢query；这里是课程正则/变量重写的具体场景。`proxy_pass_request_body on`也不能替代GET query转发。

**工程补充：** GET接口用 `@RequestParam`或查询对象绑定，不能仅用 `@RequestBody`读取这些参数。日期由前端正确编码，反向代理路径和网关剥离规则都要查看实际请求。`nginx -t`及reload只是建议验收步骤，本篇没有修改或重载Nginx。


## 22. 与行为、热点和统计计数的衔接

第09篇的行为记录、第10/11篇的热点评分和本篇评论数据不是同一张数据表。MongoDB评论和回复是内容记录，文章计数是展示/统计指标，热点评分是进一步计算出的推荐结果。

| 指标 | 对应目标 | 易混淆点 |
|---|---|---|
| 评论likes | 某条评论 | 不等于文章likes |
| 回复likes | 某条回复 | 不等于父评论likes |
| 父评论reply | 某条评论下回复数量 | 不等于文章评论数 |
| 文章comment/comments | 某篇文章 | 是否含回复需约定 |
| 统计likesNum/collectNum | 当前作者过滤后的文章集合 | 不应汇总评论/回复点赞 |

**工程补充：** 若评论创建、删除或批量关闭需要调整文章指标，可通过可靠事件同步文章服务，再供统计/热点使用。定义事件ID、文章ID、动作及计数口径，避免消费重试重复加减。

Mongo写入成功但消息发送失败、计数更新成功但请求超时、消费者重复处理，都可能造成漂移。采用事务边界内的事件记录、幂等消费者、重试与定期对账等方式处理。Mongo和MySQL之间不会因一个普通 `@Transactional`自动成为原子事务。

关闭评论后的热度是否保留历史互动，和页面现存评论数是否归零，是两个产品口径；不能只删Mongo文档就认定热点和统计已经全部同步。本篇说明边界，不声称课程提供了完整同步实现。


## 23. 两天开发顺序建议

实战说明只要求按组、两天完成全部需求，没有逐日模块划分。以下按依赖关系安排：

| 阶段 | 工作 | 可验收结果 |
|---|---|---|
| 第一天准备 | 核对发布文章ID、用户域、Mongo映射、前端版本、路由 | 身份和文章关联可追踪 |
| 第一天评论 | 六个APP接口、文本检测、游标、点赞幂等 | 评论→回复→点赞→刷新回显完整闭环 |
| 第二天管理 | 自媒体权限、文章概览、详情、回复/点赞、删除、开关 | 作者A可管理自身文章，作者B越权失败 |
| 第二天统计 | 定义日期口径、两个GET接口、Nginx参数 | 汇总和分页范围一致 |
| 第二天验收 | 并发、失败恢复、计数对账、前后端联调 | 清理无孤儿、重试不重复、指标可解释 |

小组分工可以按APP评论、自媒体管理、统计及联调划分，但先共同确定ID映射、身份协议、ResponseResult、计数语义和日期格式。只分接口、不统一协议会在最后联调时反复返工。

先按需求完成最小业务闭环，再验证一致性与边界。架构图、类名和接口数量本身都不能代替业务验收。


## 24. 验收与测试清单（待实现后执行）

下表是学习用测试计划，**没有启动业务服务执行**。

| 范围 | 测试场景 | 预期 |
|---|---|---|
| 登录 | 游客发表评论/回复/点赞 | 按认证规则拒绝写入 |
| 内容 | 空白、140字、141字、emoji边界、检测拒绝/超时 | 前后端规则一致，不把超时当成功 |
| 目标 | 不存在/非法评论ID、已关闭/不可评论文章 | 明确失败，不生成孤儿数据 |
| 评论分页 | 0条、10条、11条、相同毫秒、多页中插入新评论 | 排序稳定，限制被记录并有处理 |
| 回显 | U赞过一条，V未赞；刷新/翻页 | 当前用户状态独立 |
| 点赞 | 重复点赞、重复取消、并发请求、超时重试 | 一个活动关系，计数不重复/不负 |
| 回复 | 保存成功、重试、与父评论删除并发 | reply与实际有效回复一致 |
| 权限 | 作者B传入A的文章/评论/回复ID | 所有管理入口均拒绝越权 |
| 删除 | 父评论带多条回复和点赞；重复删除；中途故障 | 无孤儿，计数与关系可恢复 |
| 开关 | 有评论关闭、清理期间写入、重复关闭、再开启 | 关闭禁写且清理；旧评论不自动恢复 |
| 统计 | A/B隔离、0数据、缺失关联、空计数、超过一页 | 全量汇总与相同范围total一致 |
| 日期 | 起止倒置、缺省、非法格式、午夜/毫秒边界 | 约定格式和时区一致 |
| 代理 | GET同时携带日期、page、size | 服务端收到原参数 |
| 协议 | code=0/200差异、Long ID、repay路径、前缀 | 与实际前端和Controller匹配 |

对账建议：按父评论统计有效回复数量对比reply；按完整用户身份+目标统计活动点赞对比likes；按文章汇总评论数量对比文章指标。只查看按钮是否变色，无法证明存储和计数正确。


## 25. 常见问题排查

| 现象 | 优先检查 |
|---|---|
| Mongo有记录但查询为空 | 数据库/集合名、entryId Long类型、_id ObjectId类型、日期字段类型 |
| 导入后实体转换失败 | `_class`历史包名、当前实体映射与Spring Data配置 |
| 新评论查不到、分页漏评论 | entryId映射、minDate解析/时区、同毫秒边界、排序与过滤 |
| 回复接口404 | 是否误把comment_repay改成comment_reply；网关/Nginx前缀 |
| 点赞一次加两次 | 重复请求、无唯一关系索引、计数与关系未原子协调 |
| 取消后计数为负 | 未判断活动关系是否真实删除，重试重复减 |
| 点赞数正确但高亮错误 | 查询的是不是当前用户；0/null/1回显约定是否混用 |
| 删除后回复仍在 | 未收集子回复ID并清理回复和回复点赞关系 |
| 关闭后仍新增评论 | 后端未检查开关，或状态检查与写入间竞争未处理 |
| 作者B能操作A的评论 | 只做登录校验，没有按评论→文章→作者校验归属 |
| 统计只等于当前10条 | 错把分页结果用于全量汇总 |
| 统计缺文章或重复放大 | Wm稿件与APP文章关联、发布状态、JOIN/批量结果基数 |
| 日期筛选无效 | `$args`丢失、GET参数绑定错误、字段格式/时区不匹配 |
| 页面提示失败但HTTP200 | JSON业务code与前端判断不一致 |
| 统计与热点不一致 | 更新延迟、评论数口径、重复消息、删除/关闭未同步 |

按“请求→代理→网关→身份→服务入参→查询条件→BSON类型→写入/计数→响应”的顺序缩小范围。只改前端显示会掩盖数据漂移，应保留可关联的请求与业务ID日志，同时避免记录凭据和完整敏感内容。


## 26. 自测问答与资料覆盖

1. **`.sql`能直接导入MySQL吗？** 不能；已读内容是MongoDB脚本，包含清集合操作。
2. **评论主键与文章ID有什么区别？** 评论_id为ObjectId；文章entryId为Long，API表示与库内类型要分别处理。
3. **为什么只存likes数字不够？** 无法识别当前用户已赞，也无法防止重复请求。
4. **operation=0在所有接口都表示同一件事吗？** 不是；点赞0表示赞，评论开关0表示关闭。
5. **回复API为什么叫repay？** 原资料如此命名，必须兼容现有请求。
6. **时间游标为什么可能漏读？** 同毫秒记录用严格小于过滤会漏掉边界，需稳定排序和双字段游标等方案。
7. **`$inc`是否保证整个点赞业务原子？** 只保证单文档更新；关系和计数仍需事务或恢复机制。
8. **自媒体登录后能管理任何文章吗？** 不能，要证明目标文章属于当前作者。
9. **关闭评论只是禁用按钮吗？** 课程要求后端禁写，并清除该文章既有评论。
10. **删除父评论要处理什么？** 回复、两类点赞关系与计数也要按定义同步维护。
11. **当前累计计数能算一周新增量吗？** 不能自动还原，需要行为时间明细/聚合；按文章发布时间筛选是另一种口径。
12. **统计汇总能把当前页相加吗？** 不行，必须覆盖全部符合作者和时间条件的文章。
13. **为何新前端需要 `$args`？** 课程用变量重构代理URI，需要明确携带GET query。
14. **HTTP200和响应code=200是同一层吗？** 不是；原文不同模块存在业务code示例差异，须以实际协议联调。

### 已读资料覆盖

| 资料文件 | 本篇对应内容 |
|---|---|
| `01-说明/实战说明.md` | 两天分组安排、三个模块、MongoDB、新前端与Nginx参数 |
| `01-文章评论[实战]/需求文档/文章评论-需求说明.md` | 登录、140字、检测、评论/回复、排序与回显 |
| `01-文章评论[实战]/接口文档/文章评论-接口文档.md` | 六个APP接口、DTO、operation和响应 |
| `02-自媒体评论管理[实战]/需求说明/自媒体-评论管理-需求说明.md` | 管理列表、开关/清理、详情、作者操作、自媒体入口 |
| `02-自媒体评论管理[实战]/接口文档/评论管理-接口文档.md` | 七个管理接口及返回结构 |
| `03-自媒体图文统计[实战]/需求说明/自媒体-图文数据统计.md` | 当前作者、时间筛选与文章指标 |
| `03-自媒体图文统计[实战]/接口文档/自媒体图文统计-接口文档.md` | 两个GET接口、汇总/分页字段 |
| `03-资料/评论的表-导入mongodb中/leadnews-comment.sql` | 四个集合、BSON类型、关系与历史_class |

前三个模块目录位于 `day13-项目实战[2天]/02-需求/`，公共资料与说明位于该day13根目录。本篇未读取参考工程或图片，也未执行数据库脚本；索引、事务/补偿、权限细化、稳定游标、统计口径及验收计划均已标为工程补充。
